## Data collection and loading


In [1]:
%pip install yfinance pandas_datareader kaggle

In [2]:
import yfinance as yf
import pandas_datareader.data as web
import pandas as pd

start_date = '2000-01-01'
end_date = '2024-12-31'

# Download S&P 500 data (using SPY ETF as a proxy)
spy_data = yf.download('SPY', start=start_date, end=end_date)

# Download economic data from FRED
# Example FRED series IDs (you might need to find more relevant ones)
fred_series = ['FEDFUNDS', 'CPIAUCSL', 'UNRATE']
fred_data = web.DataReader(fred_series, 'fred', start=start_date, end=end_date)

# Ensure dataframes have datetime index
spy_data.index = pd.to_datetime(spy_data.index)
fred_data.index = pd.to_datetime(fred_data.index)

# Display the first few rows of the dataframes
display(spy_data.head())
display(fred_data.head())

/tmp/ipython-input-3881911617.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy_data = yf.download('SPY', start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2000-01-03,91.887749,93.664693,90.900558,93.664693,8164300
2000-01-04,88.294395,91.019044,88.225292,90.683399,8089800
2000-01-05,88.452362,89.419810,86.714905,88.412874,12177900
2000-01-06,87.030785,89.400044,87.030785,88.215414,6227200
2000-01-07,92.085243,92.085243,88.491865,88.649816,8066500


,FEDFUNDS,CPIAUCSL,UNRATE
DATE,,,
2000-01-01,5.45,169.3,4.0
2000-02-01,5.73,170.0,4.1
2000-03-01,5.85,171.0,4.0
2000-04-01,6.02,170.9,3.8
2000-05-01,6.27,171.2,4.0


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [3]:
# Inspect the loaded DataFrames
print("SPY Data Info:")
spy_data.info()
print("\nFRED Data Info:")
fred_data.info()

SPY Data Info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6288 entries, 2000-01-03 to 2024-12-30
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, SPY)   6288 non-null   float64
 1   (High, SPY)    6288 non-null   float64
 2   (Low, SPY)     6288 non-null   float64
 3   (Open, SPY)    6288 non-null   float64
 4   (Volume, SPY)  6288 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 294.8 KB

FRED Data Info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 300 entries, 2000-01-01 to 2024-12-01
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   FEDFUNDS  300 non-null    float64
 1   CPIAUCSL  300 non-null    float64
 2   UNRATE    300 non-null    float64
dtypes: float64(3)
memory usage: 9.4 KB


In [8]:
import numpy as np

# Check current column names of spy_data
print(spy_data.columns)

# Assuming the column names are now flattened and look like 'Close_SPY', 'High_SPY', etc.
# Re-calculate the features using the flattened column names

# 1. Calculate the daily log returns for the 'Close' price of SPY
spy_data['Log_Returns_'] = np.log(spy_data['Close_SPY'] / spy_data['Close_SPY'].shift(1))

# 2. Calculate the 20-day rolling mean of the log returns
spy_data['Rolling_Mean_20D_'] = spy_data['Log_Returns_'].rolling(window=20).mean()

# 3. Calculate the 20-day rolling standard deviation of the log returns
spy_data['Rolling_Std_20D_'] = spy_data['Log_Returns_'].rolling(window=20).std()

# 4. Calculate the 20-day momentum of the 'Close' price
spy_data['Momentum_20D_'] = spy_data['Close_SPY'].pct_change(periods=20)

# 5. Create the regime labels based on the 20-day rolling mean of log returns
# Define a small threshold around zero for the 'neutral' regime
threshold = 0.0001
spy_data['Regime_'] = 'neutral'
spy_data.loc[spy_data['Rolling_Mean_20D_'] > threshold, 'Regime_'] = 'bull'
spy_data.loc[spy_data['Rolling_Mean_20D_'] < -threshold, 'Regime_'] = 'bear'

# 6. Rename the volume column to a more descriptive name
# This step might have been done already, checking if 'Volume_SPY' exists before renaming.
if 'Volume_SPY' in spy_data.columns:
    spy_data.rename(columns={'Volume_SPY': 'SPY_Volume'}, inplace=True)


# 7. Merge the spy_data and fred_data DataFrames on their index (Date).
# Handle missing values by forward filling.
merged_data = spy_data.merge(fred_data, left_index=True, right_index=True, how='left')
merged_data.fillna(method='ffill', inplace=True)

# 8. Calculate additional features from the FRED data (example: change in fed funds rate)
merged_data['FEDFUNDS_Change'] = merged_data['FEDFUNDS'].diff()

# 9. Drop original columns that are no longer needed (original price columns)
# Assuming the columns are now flattened as confirmed in the print statement
columns_to_drop = ['Close_SPY', 'High_SPY', 'Low_SPY', 'Open_SPY']
merged_data.drop(columns=columns_to_drop, inplace=True)


display(merged_data.head())
display(merged_data.tail())
display(merged_data.info())

Index(['Close_SPY', 'High_SPY', 'Low_SPY', 'Open_SPY', 'Volume_SPY',
       'Log_Returns_', 'Rolling_Mean_20D_', 'Rolling_Std_20D_',
       'Momentum_20D_', 'Regime_'],
      dtype='object')


/tmp/ipython-input-2962320206.py:37: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_data.fillna(method='ffill', inplace=True)


,SPY_Volume,Log_Returns_,Rolling_Mean_20D_,Rolling_Std_20D_,Momentum_20D_,Regime_,FEDFUNDS,CPIAUCSL,UNRATE,FEDFUNDS_Change
Date,,,,,,,,,,
2000-01-03,8164300,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN
2000-01-04,8089800,-0.039891,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN
2000-01-05,12177900,0.001787,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN
2000-01-06,6227200,-0.016202,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN
2000-01-07,8066500,0.056453,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN


,SPY_Volume,Log_Returns_,Rolling_Mean_20D_,Rolling_Std_20D_,Momentum_20D_,Regime_,FEDFUNDS,CPIAUCSL,UNRATE,FEDFUNDS_Change
Date,,,,,,,,,,
2024-12-23,57635800,0.005970,0.000099,0.008539,0.001984,neutral,4.64,316.449,4.2,0.0
2024-12-24,33160100,0.011054,0.000482,0.008860,0.009696,bull,4.64,316.449,4.2,0.0
2024-12-26,41219100,0.000067,0.000225,0.008790,0.004518,bull,4.64,316.449,4.2,0.0
2024-12-27,64969300,-0.010582,-0.000152,0.009094,-0.003035,bear,4.64,316.449,4.2,0.0
2024-12-30,56578800,-0.011477,-0.001035,0.009301,-0.020497,bear,4.64,316.449,4.2,0.0


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6288 entries, 2000-01-03 to 2024-12-30
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   SPY_Volume         6288 non-null   int64  
 1   Log_Returns_       6287 non-null   float64
 2   Rolling_Mean_20D_  6268 non-null   float64
 3   Rolling_Std_20D_   6268 non-null   float64
 4   Momentum_20D_      6268 non-null   float64
 5   Regime_            6288 non-null   object 
 6   FEDFUNDS           6268 non-null   float64
 7   CPIAUCSL           6268 non-null   float64
 8   UNRATE             6268 non-null   float64
 9   FEDFUNDS_Change    6267 non-null   float64
dtypes: float64(8), int64(1), object(1)
memory usage: 540.4+ KB


None

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Handle any remaining missing values using forward fill
merged_data.fillna(method='ffill', inplace=True)
# After forward fill, if there are still NaNs (e.g., at the very beginning of the series),
# use backward fill or drop rows. Backward fill is often appropriate for time series.
merged_data.fillna(method='bfill', inplace=True)
# If any NaNs still exist, which is unlikely after ffill and bfill on time series,
# let's check and report.
if merged_data.isnull().sum().sum() > 0:
    print("Warning: Still missing values after fillna.")
    print(merged_data.isnull().sum())

# 2. Define features (X) and target variables (y)
# Features: All columns except Log_Returns_ and Regime_
features = merged_data.drop(columns=['Log_Returns_', 'Regime_'])
# Target variables: Log returns for regression, Regime_ for classification
target_returns = merged_data['Log_Returns_']
target_regime = merged_data['Regime_']

# Convert regime labels to numerical format for classification
# Using factorize for simplicity, assuming the order doesn't matter for the model
# If order matters, explicit mapping might be better.
merged_data['Regime_Numerical'] = pd.factorize(merged_data['Regime_'])[0]
target_regime_numerical = merged_data['Regime_Numerical']


# 3. Identify numerical features for scaling
# Exclude target variables and non-numeric regime column
numerical_features_columns = features.select_dtypes(include=np.number).columns.tolist()


# 4. Initialize a scaler
scaler = StandardScaler()

# 5. Apply the scaler to the identified numerical feature columns
merged_data[numerical_features_columns] = scaler.fit_transform(merged_data[numerical_features_columns])

# Display scaled data head to verify
print("\nScaled Data Head:")
display(merged_data.head())

# 6. Define the start and end dates for the splits
train_start, train_end = '2000-01-01', '2015-12-31'
val_start, val_end = '2016-01-01', '2019-12-31'
test_start, test_end = '2020-01-01', '2024-12-31'

# Split data into initial train, validation, and the full test set period
train_data = merged_data.loc[train_start:train_end]
val_data = merged_data.loc[val_start:val_end]
test_data_full = merged_data.loc[test_start:test_end]


# Define features and targets for the initial splits
X_train = train_data[numerical_features_columns] # Using scaled features
y_train_returns = train_data['Log_Returns_']
y_train_regime = train_data['Regime_Numerical']

X_val = val_data[numerical_features_columns] # Using scaled features
y_val_returns = val_data['Log_Returns_']
y_val_regime = val_data['Regime_Numerical']

# 7. Implement a walk-forward splitting strategy for the test set (2020-2024)
# We will generate walk-forward splits later during the model training/evaluation phase
# For now, we just need the full test data frame.

# Print shapes of the initial splits to verify
print("\nShapes of initial splits:")
print("X_train shape:", X_train.shape)
print("y_train_returns shape:", y_train_returns.shape)
print("y_train_regime shape:", y_train_regime.shape)
print("X_val shape:", X_val.shape)
print("y_val_returns shape:", y_val_returns.shape)
print("y_val_regime shape:", y_val_regime.shape)
print("test_data_full shape:", test_data_full.shape)


Scaled Data Head:


/tmp/ipython-input-3354120626.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_data.fillna(method='ffill', inplace=True)
/tmp/ipython-input-3354120626.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_data.fillna(method='bfill', inplace=True)


,SPY_Volume,Log_Returns_,Rolling_Mean_20D_,Rolling_Std_20D_,Momentum_20D_,Regime_,FEDFUNDS,CPIAUCSL,UNRATE,FEDFUNDS_Change,Regime_Numerical
Date,,,,,,,,,,,
2000-01-03,-1.076551,-0.039891,-0.786855,1.671874,-0.819314,neutral,1.878194,-1.570823,-0.813241,0.003339,0
2000-01-04,-1.077370,-0.039891,-0.786855,1.671874,-0.819314,neutral,1.878194,-1.570823,-0.813241,0.003339,0
2000-01-05,-1.032435,0.001787,-0.786855,1.671874,-0.819314,neutral,1.878194,-1.570823,-0.813241,0.003339,0
2000-01-06,-1.097844,-0.016202,-0.786855,1.671874,-0.819314,neutral,1.878194,-1.570823,-0.813241,0.003339,0
2000-01-07,-1.077626,0.056453,-0.786855,1.671874,-0.819314,neutral,1.878194,-1.570823,-0.813241,0.003339,0



Shapes of initial splits:
X_train shape: (4025, 8)
y_train_returns shape: (4025,)
y_train_regime shape: (4025,)
X_val shape: (1006, 8)
y_val_returns shape: (1006,)
y_val_regime shape: (1006,)
test_data_full shape: (1257, 11)


In [19]:
print("Value counts for 'Regime_' in train_data:")
display(train_data['Regime_'].value_counts())

Value counts for 'Regime_' in train_data:


,count
Regime_,
bull,2369
bear,1492
neutral,164
